# Reliability block: two labellers on the 100-episode block

Pre-registered in `data/DATA_RULES.md` (31 Oct 2026, before labelling): report, whatever the values,
inter-coder agreement on both gates and the structure dimensions on positions 1–100 of
`queue_order.csv`, and the lexical filter's recall estimated on the 46 unflagged episodes.

Inputs: one clean label file per coder (`labels/<code>.csv`, codes only), the current episode file
(sentence unit, codebook v1.2), the queue order, Jev's silver labels. Coders are **A**, **B** (the
two who completed the block) and **C** (97/100 at the time of writing).

Caveats fixed in advance of reading the numbers:
- A label made before the cleaned sentence-unit text went live (31 Oct 2026 10:40 UTC) was made on a
  different text and unit. Every number is reported on all latest labels and on current-text labels only.
- Gwet's AC1 and PABAK are shown beside κ because "yes" is rare among unflagged episodes (κ's prevalence problem).
- 95% intervals: bootstrap over transcripts (participants), since episodes from one participant are not independent.

In [1]:
import json, sqlite3
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score

FW = Path.cwd().parent if Path.cwd().name == "benchmark" else Path.cwd()
LAB = FW.parent / "xai-contestation-labelling"
FILES = {"A": LAB / "labels" / "c_433b209090.csv", "B": LAB / "labels" / "c_bf6244ebdc.csv", "C": LAB / "labels" / "c_01534ba437.csv"}
CURRENT_TEXT_FROM = "2026-10-31 10:40:00"   # UTC, when the cleaned sentence-unit version was pushed
B_SEED, B_N = 20261103, 2000

eps = pd.read_csv(FW / "data" / "episodes_to_label.csv", keep_default_na=False).set_index("episode_id")
order = pd.read_csv(FW / "data" / "queue_order.csv").set_index("episode_id")
BLOCK = order[order.queue_pos <= 100].index
DIMS = ["presence", "target", "interaction_act", "grounds", "expected_response"]

frames = []
for letter, f in FILES.items():
    d = pd.read_csv(f, dtype=str, keep_default_na=False)
    d["is_skipped"] = pd.to_numeric(d.is_skipped).astype(int); d["time_spent_s"] = pd.to_numeric(d.time_spent_s, errors="coerce")
    d = d.replace({"": None})
    print(f"coder {letter}: {len(d)} saves")
    frames.append(d.assign(coder=letter))
raw = pd.concat(frames, ignore_index=True).drop(columns=["annotator_code"])
raw["current_text"] = raw.created_at >= CURRENT_TEXT_FROM
lab = (raw[raw.is_skipped == 0].sort_values("created_at")
       .drop_duplicates(["coder", "episode_id"], keep="last").set_index(["coder", "episode_id"]))
print(lab.groupby("coder").agg(episodes=("is_contestation", "size"), current_text=("current_text", "sum")))

coder A: 205 saves
coder B: 183 saves
coder C: 114 saves
       episodes  current_text
coder                        
A           182           169
B           171           161
C           104           104


## 1. Coverage, pace, prevalence

In [2]:
rows = []
for c in "ABC":
    L = lab.loc[c]
    inb = L[L.index.isin(BLOCK)]
    rows.append({"coder": c, "labelled": len(L), "block labelled": len(inb),
                 "block on current text": int(inb.current_text.sum()),
                 "block contestation yes": int((inb.is_contestation == "yes").sum()),
                 "block request yes": int((inb.explanation_request == "yes").sum()),
                 "block request missing": int(inb.explanation_request.isna().sum()),
                 "median s/episode": round(L.time_spent_s.median(), 1),
                 "episodes < 10 s": int((L.time_spent_s < 10).sum())})
pd.DataFrame(rows).set_index("coder")

,labelled,block labelled,block on current text,block contestation yes,block request yes,block request missing,median s/episode,episodes < 10 s
coder,,,,,,,,
A,182,100,94,29,22,0,40.2,22
B,171,100,100,41,34,0,56.9,6
C,104,100,100,28,24,0,39.9,17


## 2. Agreement on the block

In [3]:
def ac1(a, b):
    a, b = np.asarray(a), np.asarray(b)
    pa = (a == b).mean()
    cats = np.union1d(a, b)
    pi = np.array([((a == k).mean() + (b == k).mean()) / 2 for k in cats])
    pe = (pi * (1 - pi)).sum() / (len(cats) - 1) if len(cats) > 1 else 1.0
    return (pa - pe) / (1 - pe) if pe < 1 else np.nan

def stats(a, b):
    a, b = np.asarray(a), np.asarray(b)
    pa = (a == b).mean()
    k = cohen_kappa_score(a, b) if len(np.union1d(a, b)) > 1 else np.nan
    return {"n": len(a), "agreement": pa, "kappa": k, "AC1": ac1(a, b), "PABAK": 2 * pa - 1}

def boot(df, col_a, col_b, metric, groups):
    rng = np.random.default_rng(B_SEED)
    idx = [np.flatnonzero(df[groups].values == t) for t in df[groups].unique()]
    a_all, b_all = df[col_a].values, df[col_b].values
    out = []
    for _ in range(B_N):
        rows = np.concatenate([idx[i] for i in rng.integers(len(idx), size=len(idx))])
        v = stats(a_all[rows], b_all[rows])[metric]
        if not np.isnan(v):
            out.append(v)
    return np.percentile(out, [2.5, 97.5]) if out else (np.nan, np.nan)

pair = lab.loc["A"].add_suffix("_A").join(lab.loc["B"].add_suffix("_B"), how="inner")
pair = pair[pair.index.isin(BLOCK)].join(eps[["transcript_id", "participant_group", "explanation_format", "sample_batch"]])
pair["both_current"] = pair.current_text_A & pair.current_text_B

def table(gate, subset_name, p):
    p = p.dropna(subset=[f"{gate}_A", f"{gate}_B"])
    s = stats(p[f"{gate}_A"], p[f"{gate}_B"])
    lo, hi = boot(p, f"{gate}_A", f"{gate}_B", "kappa", "transcript_id")
    alo, ahi = boot(p, f"{gate}_A", f"{gate}_B", "AC1", "transcript_id")
    return {"gate": gate, "labels": subset_name, **{k: round(v, 3) if isinstance(v, float) else v for k, v in s.items()},
            "kappa 95% CI": f"{lo:.2f}–{hi:.2f}", "AC1 95% CI": f"{alo:.2f}–{ahi:.2f}"}

res = []
for gate in ["is_contestation", "explanation_request"]:
    res.append(table(gate, "all block", pair))
    res.append(table(gate, "both on current text", pair[pair.both_current]))
    for b in ["flagged", "random_unflagged"]:
        res.append(table(gate, f"all block, {b}", pair[pair.sample_batch == b]))
R = pd.DataFrame(res).set_index(["gate", "labels"])
R

n  agreement  kappa    AC1  \
gate                labels                                                      
is_contestation     all block                    100      0.780  0.524  0.596   
                    both on current text          94      0.766  0.502  0.565   
                    all block, flagged            54      0.667  0.372  0.337   
                    all block, random_unflagged   46      0.913  0.618  0.888   
explanation_request all block                    100      0.820  0.561  0.698   
                    both on current text          94      0.809  0.541  0.676   
                    all block, flagged            54      0.815  0.608  0.653   
                    all block, random_unflagged   46      0.826  0.412  0.756   

                                                 PABAK kappa 95% CI AC1 95% CI  
gate                labels                                                      
is_contestation     all block                    0.560    0.32–0.70  0.35–0.79  
                    both on current text         0.532    0.30–0.68  0.33–0.78  
                    all block, flagged           0.333    0.14–0.58  0.07–0.59  
                    all block, random_unflagged  0.826    0.16–0.92  0.72–0.98  
explanation_request all block                    0.640    0.38–0.72  0.52–0.84  
                    both on current text         0.617    0.35–0.71  0.49–0.83  
                    all block, flagged           0.630    0.38–0.83  0.41–0.86  
                    all block, random_unflagged  0.652    0.11–0.66  0.51–0.90

In [4]:
for gate in ["is_contestation", "explanation_request"]:
    p = pair.dropna(subset=[f"{gate}_A", f"{gate}_B"])
    print(gate); print(pd.crosstab(p[f"{gate}_A"], p[f"{gate}_B"], rownames=["A"], colnames=["B"])); print()

is_contestation
B    no  yes
A           
no   54   17
yes   5   24

explanation_request
B    no  yes
A           
no   63   15
yes   3   19



## 2b. Any push-back (dispute or request), three coders, by sample half

The block over-samples the boundary (54 filter-selected episodes). The system's first decision is
whether the participant pushes back at all; the second is whether it is a dispute or a request.

In [5]:
def fleiss_yes(cols):
    M = np.array([[sum(c[i] == "yes" for c in cols), sum(c[i] != "yes" for c in cols)] for i in range(len(cols[0]))])
    n = M.sum(1)[0]; P = ((M ** 2).sum(1) - n) / (n * (n - 1)); pj = M.sum(0) / M.sum()
    return (P.mean() - (pj ** 2).sum()) / (1 - (pj ** 2).sum())
batch = eps.loc[BLOCK, "sample_batch"]
rows = []
for name, ids in [("block", list(BLOCK)), ("flagged", list(BLOCK[batch == "flagged"])), ("random_unflagged", list(BLOCK[batch == "random_unflagged"]))]:
    con = {k: lab.loc[k].loc[ids, "is_contestation"].values for k in "ABC"}
    anyp = {k: np.where((lab.loc[k].loc[ids, "is_contestation"] == "yes") | (lab.loc[k].loc[ids, "explanation_request"] == "yes"), "yes", "no") for k in "ABC"}
    for lbl, d in [("contestation", con), ("any push-back", anyp)]:
        pw = [cohen_kappa_score(d[a], d[b]) for a, b in [("A", "B"), ("A", "C"), ("B", "C")]]
        ag = [(d[a] == d[b]).mean() for a, b in [("A", "B"), ("A", "C"), ("B", "C")]]
        rows.append({"set": name, "label": lbl, "n": len(ids), "pairwise κ": f"{min(pw):.2f}–{max(pw):.2f}",
                     "agreement": f"{min(ag):.2f}–{max(ag):.2f}", "Fleiss κ": round(fleiss_yes([d[k] for k in "ABC"]), 2),
                     "AC1 A–B": round(ac1(d["A"], d["B"]), 2)})
    tot = gate = 0
    for a, b in [("A", "B"), ("A", "C"), ("B", "C")]:
        dis = [i for i in range(len(ids)) if con[a][i] != con[b][i]]; tot += len(dis)
        gate += sum(1 for i in dis if anyp[a][i] == "yes" and anyp[b][i] == "yes")
    print(f"{name}: contestation disagreements over 3 pairs = {tot}, of which both saw push-back and differed only on dispute vs request = {gate} ({gate / max(tot, 1):.0%})")
pd.DataFrame(rows).set_index(["set", "label"])

block: contestation disagreements over 3 pairs = 68, of which both saw push-back and differed only on dispute vs request = 31 (46%)
flagged: contestation disagreements over 3 pairs = 54, of which both saw push-back and differed only on dispute vs request = 26 (48%)
random_unflagged: contestation disagreements over 3 pairs = 14, of which both saw push-back and differed only on dispute vs request = 5 (36%)


n pairwise κ  agreement  Fleiss κ  AC1 A–B
set              label                                                      
block            contestation   100  0.44–0.52  0.77–0.78      0.48     0.60
                 any push-back  100  0.50–0.68  0.75–0.84      0.60     0.68
flagged          contestation    54  0.31–0.37  0.67–0.67      0.33     0.34
                 any push-back   54  0.43–0.71  0.72–0.87      0.57     0.77
random_unflagged contestation    46  0.48–0.62  0.89–0.91      0.55     0.89
                 any push-back   46  0.51–0.57  0.78–0.80      0.53     0.68

**Phase 1 reference** (same construct, line unit, codebook v1.0, 36 double-coded episodes, first author +
one coder): first-pass κ = 0.222, raw agreement 0.611; κ ≈ 0.28 after the matched-context recheck.

## 3. Structure dimensions, on episodes both coded as contestation

In [6]:
both_yes = pair[(pair.is_contestation_A == "yes") & (pair.is_contestation_B == "yes")]
rows = []
for d in DIMS:
    s = stats(both_yes[f"{d}_A"].fillna(""), both_yes[f"{d}_B"].fillna(""))
    rows.append({"dimension": d, **{k: round(v, 3) if isinstance(v, float) else v for k, v in s.items()},
                 "values used (A/B)": f"{both_yes[f'{d}_A'].nunique()}/{both_yes[f'{d}_B'].nunique()}"})
pd.DataFrame(rows).set_index("dimension")

,n,agreement,kappa,AC1,PABAK,values used (A/B)
dimension,,,,,,
presence,24,0.417,0.034,0.193,-0.167,3/2
target,24,0.208,0.067,0.076,-0.583,5/5
interaction_act,24,0.333,0.101,0.241,-0.333,7/6
grounds,24,0.583,0.383,0.540,0.167,7/6
expected_response,24,0.458,0.283,0.337,-0.083,5/5


## 3b. Why the dimensions disagree, and the level at which the headline claim is reliable

The two coders use different vocabularies rather than judging the same scale differently. Checked
before computing (one composite only, stated in advance): under codebook v1.1 an objection to *how*
the explanation is shown can be coded either as contestation with `target = explanation
representation` or as `explanation_request = yes`. The composite **form objection** = request = yes
OR (contestation = yes AND target = representation); **prediction dispute** = contestation = yes AND
target = prediction; **content dispute** = contestation = yes with any other target.

In [7]:
rep = pair[(pair.is_contestation_B == "yes") & (pair.target_B == "explanation representation")]
print("B's 'explanation representation' contestations, as coded by A:")
print(pd.crosstab(rep.is_contestation_A, rep.explanation_request_A, rownames=["A contestation"], colnames=["A request"]))
ev = pair[(pair.is_contestation_A == "yes") & (pair.target_A == "evidence")]
print("\nA's 'evidence' contestations: B's target when B also said yes:", ev[ev.is_contestation_B == "yes"].target_B.value_counts().to_dict())
for c in "AB":
    pair[f"form_{c}"] = (pair[f"explanation_request_{c}"] == "yes") | ((pair[f"is_contestation_{c}"] == "yes") & (pair[f"target_{c}"] == "explanation representation"))
    pair[f"pred_{c}"] = (pair[f"is_contestation_{c}"] == "yes") & (pair[f"target_{c}"] == "prediction")
    pair[f"content_{c}"] = (pair[f"is_contestation_{c}"] == "yes") & ~pair[f"target_{c}"].isin(["prediction", "explanation representation"])
    pair[f"cat_{c}"] = np.select([pair[f"pred_{c}"], pair[f"form_{c}"], pair[f"content_{c}"]], ["prediction", "form", "content"], "none")
rows = []
for c in "AB":
    rows.append({"coder": c, "form objection": int(pair[f"form_{c}"].sum()), "content dispute": int(pair[f"content_{c}"].sum()),
                 "prediction dispute": int(pair[f"pred_{c}"].sum())})
print("\nblock (n=100):"); print(pd.DataFrame(rows).set_index("coder"))
for name, col in [("form objection", "form"), ("4-way form/content/prediction/none", "cat")]:
    s = stats(pair[f"{col}_A"], pair[f"{col}_B"]); lo, hi = boot(pair, f"{col}_A", f"{col}_B", "kappa", "transcript_id")
    print(f"{name}: agreement={s['agreement']:.2f} kappa={s['kappa']:.2f} (95% CI {lo:.2f}–{hi:.2f}) AC1={s['AC1']:.2f}")
print(pd.crosstab(pair.cat_A, pair.cat_B, rownames=["A"], colnames=["B"]))

B's 'explanation representation' contestations, as coded by A:
A request       no  yes
A contestation         
no               2    9
yes              2    1

A's 'evidence' contestations: B's target when B also said yes: {'reasoning': 11, 'input data': 2, 'prediction': 1, 'explanation representation': 1}

block (n=100):
       form objection  content dispute  prediction dispute
coder                                                     
A                  23               27                   1
B                  35               26                   1


form objection: agreement=0.80 kappa=0.52 (95% CI 0.32–0.70) AC1=0.66


4-way form/content/prediction/none: agreement=0.74 kappa=0.58 (95% CI 0.42–0.74) AC1=0.67
B           content  form  none  prediction
A                                          
content          12     6     4           1
form              2    19     2           0
none              1     9    43           0
prediction        0     1     0           0


## 3c. The 62 overlapping episodes beyond the block (all flagged)

In [8]:
allp = lab.loc["A"].add_suffix("_A").join(lab.loc["B"].add_suffix("_B"), how="inner").join(eps[["transcript_id", "sample_batch"]])
allp["in_block"] = allp.index.isin(BLOCK)
rows = []
for name, sub in [("block", allp[allp.in_block]), ("beyond block (flagged only)", allp[~allp.in_block]),
                  ("all flagged (block + beyond)", allp[allp.sample_batch == "flagged"]), ("all overlap", allp)]:
    s = stats(sub.is_contestation_A, sub.is_contestation_B); lo, hi = boot(sub, "is_contestation_A", "is_contestation_B", "kappa", "transcript_id")
    rows.append({"set": name, "n": len(sub), "agreement": round(s["agreement"], 3), "kappa": round(s["kappa"], 3), "kappa 95% CI": f"{lo:.2f}–{hi:.2f}", "AC1": round(s["AC1"], 3)})
pd.DataFrame(rows).set_index("set")

,n,agreement,kappa,kappa 95% CI,AC1
set,,,,,
block,100,0.780,0.524,0.32–0.70,0.596
beyond block (flagged only),62,0.677,0.361,0.09–0.62,0.365
all flagged (block + beyond),116,0.672,0.371,0.17–0.57,0.346
all overlap,162,0.741,0.462,0.27–0.65,0.509


## 3d. Coarse levels of the dimensions (one pass, all reported)

Post-hoc but definition-based: each dimension is collapsed to the level named by the construct
definition (target: conclusion / account / form / authority), by the coder-uncertainty reading of
"ambiguous" (presence: explicit / implicit), by the four calls in the definition (act: ask / assert
wrong / propose / uncertainty), by Phase 2's verifiability split (grounds: checkable / not), and by
the Hold–Revise–Surface mapping (expected response: hold- / revise- / surface-type). Computed once;
nothing else was tried.

In [9]:
COARSE = {
 "target": {"prediction": "conclusion", "reasoning": "account", "evidence": "account", "input data": "account",
            "explanation representation": "form", "system competence": "authority"},
 "presence": {"explicit": "explicit", "implicit": "implicit", "ambiguous": "implicit"},
 "interaction_act": {"request for justification": "ask", "direct challenge": "assert wrong", "rejection": "assert wrong",
                     "counterexample": "assert wrong", "correction": "propose", "alternative proposal": "propose",
                     "request to acknowledge uncertainty": "uncertainty"},
 "grounds": {"presentation ambiguity": "checkable", "internal inconsistency": "checkable", "missing evidence": "checkable",
             "data-quality concern": "checkable", "causal implausibility": "checkable",
             "domain knowledge": "not checkable", "technical knowledge": "not checkable", "prior experience": "not checkable"},
 "expected_response": {"revise": "revise-type", "incorporate correction": "revise-type", "justify": "hold-type",
                       "expose conflicting evidence": "hold-type", "provide additional information": "hold-type",
                       "acknowledge uncertainty": "surface-type", "defer to human expertise": "surface-type"},
}
allp2 = lab.loc["A"].add_suffix("_A").join(lab.loc["B"].add_suffix("_B"), how="inner").join(eps[["transcript_id"]])
rows = []
for name, sub in [("block", both_yes), ("all overlap", allp2[(allp2.is_contestation_A == "yes") & (allp2.is_contestation_B == "yes")])]:
    for d, m in COARSE.items():
        a, b = sub[f"{d}_A"].fillna(""), sub[f"{d}_B"].fillna("")
        ca, cb = a.map(m).fillna("?"), b.map(m).fillna("?")
        lo, hi = boot(sub.assign(ca=ca.values, cb=cb.values), "ca", "cb", "kappa", "transcript_id")
        rows.append({"set": name, "dimension": d, "n": len(sub), "fine κ": round(cohen_kappa_score(a, b), 2),
                     "coarse agreement": round((ca == cb).mean(), 2), "coarse κ": round(cohen_kappa_score(ca, cb), 2),
                     "coarse κ 95% CI": f"{lo:.2f}–{hi:.2f}", "coarse AC1": round(ac1(ca, cb), 2)})
pd.DataFrame(rows).set_index(["set", "dimension"])

n  fine κ  coarse agreement  coarse κ  \
set         dimension                                                   
block       target             24    0.07              0.79      0.11   
            presence           24    0.03              0.62      0.28   
            interaction_act    24    0.10              0.62      0.26   
            grounds            24    0.38              0.88      0.68   
            expected_response  24    0.28              0.67      0.38   
all overlap target             41    0.11              0.78      0.42   
            presence           41    0.09              0.61      0.25   
            interaction_act    41    0.05              0.71      0.38   
            grounds            41    0.33              0.90      0.69   
            expected_response  41    0.30              0.66      0.44   

                              coarse κ 95% CI  coarse AC1  
set         dimension                                      
block       target                 -0.06–0.36        0.77  
            presence                0.02–0.57        0.28  
            interaction_act        -0.09–0.57        0.55  
            grounds                 0.21–1.00        0.79  
            expected_response       0.00–0.74        0.54  
all overlap target                  0.22–0.60        0.75  
            presence                0.07–0.46        0.27  
            interaction_act         0.04–0.62        0.65  
            grounds                 0.36–0.91        0.86  
            expected_response       0.21–0.69        0.51

## 4. Lexical filter recall (estimated from the block)

Recall = share of all contestation that the lexical filter flags. With contestation rates *r_f* in the flagged
sample and *r_u* in the random unflagged sample, and pool sizes *N_f*, *N_u* (participant lines on explanation
slides; Phase 1 `sample_manifest.json`): recall = r_f·N_f / (r_f·N_f + r_u·N_u). N_u = 4,614; N_f = 254 drawable
+ 76 gold = 330. Pools count transcript lines while the block uses sentences, so this is an approximation;
a sensitivity range for N_f is shown.

In [10]:
N_U = 4614
def recall(df, rule, n_f=330):
    yes = rule(df)
    rf = yes[df.sample_batch == "flagged"].mean()
    ru = yes[df.sample_batch == "random_unflagged"].mean()
    return rf, ru, rf * n_f / (rf * n_f + ru * N_U) if (rf + ru) > 0 else np.nan

rules = {"coder A": lambda d: d.is_contestation_A == "yes",
         "coder B": lambda d: d.is_contestation_B == "yes",
         "both yes": lambda d: (d.is_contestation_A == "yes") & (d.is_contestation_B == "yes"),
         "either yes": lambda d: (d.is_contestation_A == "yes") | (d.is_contestation_B == "yes")}
rng = np.random.default_rng(B_SEED)
rows = []
for name, rule in rules.items():
    rf, ru, rc = recall(pair, rule)
    bs = []
    for _ in range(B_N):
        s = pd.concat([g.sample(len(g), replace=True, random_state=rng.integers(1e9)) for _, g in pair.groupby("sample_batch")])
        bs.append(recall(s, rule)[2])
    rows.append({"rule": name, "rate flagged": round(rf, 3), "rate unflagged": round(ru, 3),
                 "recall (N_f=330)": round(rc, 3), "95% CI": f"{np.nanpercentile(bs, 2.5):.2f}–{np.nanpercentile(bs, 97.5):.2f}",
                 "recall if N_f=254": round(recall(pair, rule, 254)[2], 3), "recall if N_f=667": round(recall(pair, rule, 667)[2], 3)})
pd.DataFrame(rows).set_index("rule")

,rate flagged,rate unflagged,recall (N_f=330),95% CI,recall if N_f=254,recall if N_f=667
rule,,,,,,
coder A,0.407,0.152,0.161,0.09–0.35,0.128,0.279
coder B,0.667,0.109,0.305,0.18–0.68,0.252,0.470
both yes,0.370,0.087,0.233,0.12–0.58,0.190,0.381
either yes,0.704,0.174,0.224,0.14–0.41,0.182,0.369


## 5. Jev vs each coder (context only)

Jev's silver labels on the 250 were made on the raw, line-unit text before cleaning, at threshold 0.17;
this is a rough check, not a benchmark result.

In [11]:
silver = pd.read_csv(FW / "data" / "silver_250.csv").set_index("episode_id")
rows = []
for c in "AB":
    p = pair.join(silver[["is_contestation", "p_contestation"]].rename(columns={"is_contestation": "jev"}), how="inner")
    s = stats(p[f"is_contestation_{c}"], p.jev)
    rows.append({"pair": f"coder {c} vs Jev", **{k: round(v, 3) if isinstance(v, float) else v for k, v in s.items()}})
s = stats(pair.is_contestation_A, pair.is_contestation_B)
rows.append({"pair": "coder A vs coder B", **{k: round(v, 3) if isinstance(v, float) else v for k, v in s.items()}})
pd.DataFrame(rows).set_index("pair")

,n,agreement,kappa,AC1,PABAK
pair,,,,,
coder A vs Jev,92,0.696,0.406,0.405,0.391
coder B vs Jev,92,0.761,0.525,0.522,0.522
coder A vs coder B,100,0.780,0.524,0.596,0.560


## 6. Disagreements, for adjudication (written to `benchmark/outputs/reliability_block_disagreements.csv`)

In [12]:
dis = pair[(pair.is_contestation_A != pair.is_contestation_B) | (pair.explanation_request_A != pair.explanation_request_B)]
cols = ["sample_batch", "participant_group", "explanation_format", "is_contestation_A", "is_contestation_B",
        "explanation_request_A", "explanation_request_B", "notes_A", "notes_B", "both_current"]
out = dis[cols].join(eps[["text_pl", "text_en"]])
(FW / "benchmark" / "outputs").mkdir(parents=True, exist_ok=True)
out.to_csv(FW / "benchmark" / "outputs" / "reliability_block_disagreements.csv")
print(f"{len(dis)} episodes with any gate disagreement "
      f"(contestation {int((pair.is_contestation_A != pair.is_contestation_B).sum())}, "
      f"request {int((pair.explanation_request_A != pair.explanation_request_B).sum())})")
out[["sample_batch", "is_contestation_A", "is_contestation_B", "explanation_request_A", "explanation_request_B", "text_en"]].head(40)

34 episodes with any gate disagreement (contestation 22, request 18)


,sample_batch,is_contestation_A,is_contestation_B,explanation_request_A,explanation_request_B,text_en
episode_id,,,,,,
MW_SSH_05:101,flagged,no,yes,yes,yes,I think I'm not able to identify what has a ne...
MW_SSH_05:95,flagged,no,yes,yes,yes,I'm just reading. (ns - 00:34:22) of the fruit...
MW_IT_02:35,flagged,no,yes,yes,yes,"Well, I can't really, I assume that here we're..."
PK_DE_05:128,flagged,yes,yes,yes,no,"I mean, I, I cannot fully disagree with it, be..."
MK_SSH_04:109,random_unflagged,no,no,no,yes,Analysis of the impact of features on predicti...
DR_SSH_03:326,random_unflagged,no,no,yes,no,"I did pay attention, but now, coming back to i..."
DR_SSH_02:87,flagged,no,yes,no,yes,"There are 9 such mini charts, although actuall..."
PK_DE_03:89,flagged,no,yes,yes,yes,Well exactly. Well every mushroom should have ...
MW_SSH_05:75,flagged,no,yes,no,yes,"Well, I'm wondering whether… I'm not an expert..."


## 7. Third coder (C)

Rule fixed before looking (DATA_RULES, 3 Nov 2026): a coder who did not complete the block is a
supplementary partial rater; inclusion never depends on the value. Coder C labelled under two
accounts (positions 1–31, then 1–3 and 35–100), merged into one file; 97/100 covered.

In [13]:
rows = []
for gate in ["is_contestation", "explanation_request"]:
    for x, y in [("A", "B"), ("A", "C"), ("B", "C")]:
        X, Y = lab.loc[x], lab.loc[y]
        i = X.index.intersection(Y.index).intersection(BLOCK)
        a, b = X.loc[i, gate].fillna(""), Y.loc[i, gate].fillna("")
        s = stats(a, b)
        rows.append({"gate": gate, "pair": f"{x}–{y}", "n (block)": len(i), "agreement": round(s["agreement"], 2),
                     "kappa": round(s["kappa"], 2), "AC1": round(s["AC1"], 2),
                     "yes rate": f"{(a == 'yes').mean():.2f} / {(b == 'yes').mean():.2f}"})
print(pd.DataFrame(rows).set_index(["gate", "pair"]))

def fleiss(mat):
    n = mat.sum(1)[0]; P = ((mat ** 2).sum(1) - n) / (n * (n - 1)); pj = mat.sum(0) / mat.sum()
    return (P.mean() - (pj ** 2).sum()) / (1 - (pj ** 2).sum())
cd = lab.loc["C"]
i = lab.loc["A"].index.intersection(lab.loc["B"].index).intersection(cd.index).intersection(BLOCK)
for gate in ["is_contestation", "explanation_request"]:
    L3 = pd.DataFrame({"A": lab.loc["A"].loc[i, gate], "B": lab.loc["B"].loc[i, gate], "CD": cd.loc[i, gate]}).fillna("")
    mat = np.array([[(r == "yes").sum(), (r == "no").sum()] for _, r in L3.iterrows()])
    print(f"\n{gate}: three-rater Fleiss κ = {fleiss(mat):.2f} on n={len(i)} | A–B alone on the same episodes κ = {cohen_kappa_score(L3.A, L3.B):.2f}")

                          n (block)  agreement  kappa   AC1     yes rate
gate                pair                                                
is_contestation     A–B         100       0.78   0.52  0.60  0.29 / 0.41
                    A–C         100       0.77   0.44  0.61  0.29 / 0.28
                    B–C         100       0.77   0.50  0.58  0.41 / 0.28
explanation_request A–B         100       0.82   0.56  0.70  0.22 / 0.34
                    A–C         100       0.78   0.38  0.66  0.22 / 0.24
                    B–C         100       0.78   0.47  0.63  0.34 / 0.24

is_contestation: three-rater Fleiss κ = 0.48 on n=100 | A–B alone on the same episodes κ = 0.52

explanation_request: three-rater Fleiss κ = 0.47 on n=100 | A–B alone on the same episodes κ = 0.56
